# Análisis del histórico de sondas de humedad del suelo

Este cuaderno revisa el histórico de la sonda de humedad de cada parcela: cómo está organizado, cada cuánto se lee, dónde hay huecos o valores sospechosos, qué parte del periodo del estudio cubre y en qué momentos la humedad sube de forma brusca (posibles riegos o lluvias).

**Para qué sirve.** Es la tarea de la issue #9 del sprint 3. Sus resultados alimentan el glosario de variables (`docs/variables-balance-hidrico.md`) y, más adelante, la calibración de la capacidad de campo real y el simulador.

**Qué necesita.** El histórico de sondas, guardado en la carpeta privada de Drive `TFG_PATATAS_TRABAJO/privado/historico_sondas/`.

**Qué produce.** Tablas y figuras en `TFG_PATATAS_TRABAJO/resultados/analisis_historico_sondas/`. No genera recomendaciones de riego.

**Cómo usarlo.** Ejecuta las celdas en orden en Google Colab. Cada apartado empieza con una explicación de lo que hace y de cómo interpretar su resultado.

**Privacidad.** Los identificadores de las sondas se sustituyen por alias (`sonda_1`, `sonda_2`...) antes de mostrar nada. La correspondencia se guarda solo en la carpeta privada de Drive.



## 1. Entorno, rutas y configuración

Esta celda importa las librerías, monta Google Drive y define las carpetas. Después fija los parámetros del análisis. Los umbrales (huecos, saltos, subidas) son **criterios exploratorios para revisar los datos**; no son los umbrales de la calibración de la capacidad de campo, que están pendientes de los tutores.

Si el cuaderno se ejecuta fuera de Colab (por ejemplo para pruebas), usa una carpeta local en lugar de Drive.

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    base_path = Path("/content/drive/MyDrive/TFG_PATATAS_TRABAJO")
else:
    base_path = Path.cwd() / "TFG_PATATAS_TRABAJO"

historico_path = base_path / "privado" / "historico_sondas"
privado_path = base_path / "privado"
resultados_path = base_path / "resultados" / "analisis_historico_sondas"
resultados_path.mkdir(parents=True, exist_ok=True)

ARCHIVOS = None            # None = todos los CSV o Excel de la carpeta; o una lista de nombres
COL_FECHA = "Fecha"
COL_HUMEDAD = "Humedad"
COL_TEMPERATURA = "Temperatura"
COL_SONDA = "Sonda ID"

# Alias que son una extensión incorrecta del identificador de otra sonda (se fusionan con ella).
# Los alias se asignan por orden alfabético del identificador, antes de fusionar.
FUSIONES_ALIAS = {"sonda_2": "sonda_1"}

PERIODO_INICIO = "2026-04-01"   # incluido
PERIODO_FIN = "2026-11-01"      # excluido

HUECO_HORAS = 12                # intervalo entre lecturas a partir del cual se considera hueco
SALTO_MAX = 10.0                # puntos de % vol. entre dos lecturas consecutivas (exploratorio)
VENTANA_SUBIDA_H = 12           # ventana para medir una subida (horas)
UMBRAL_SUBIDA = 2.0             # puntos de % vol. que debe subir la humedad (exploratorio)
SEPARACION_EPISODIOS_H = 12     # horas sin subida para separar dos episodios
VENTANA_POST_PICO_H = (24, 48)  # ventana posterior al pico que usará la calibración de la CC

## 2. Carga de los archivos

Se leen todos los archivos de la carpeta (o los que indique `ARCHIVOS`). Los CSV se leen detectando el separador automáticamente, y los Excel con su primera hoja. Cada fila recuerda de qué archivo viene, con un alias (`archivo_1`, `archivo_2`...) para no mostrar nombres reales.

**Qué comprobar:** que el número de archivos y de filas coincide con lo que esperas. Si la carpeta está vacía, el cuaderno se detiene con un aviso.

In [ ]:
EXTENSIONES = {".csv", ".txt", ".xlsx", ".xls"}

def leer_archivo(ruta):
    """Lee un CSV o un Excel y devuelve una tabla."""
    if ruta.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(ruta)
    return pd.read_csv(ruta, sep=None, engine="python", encoding_errors="replace")

if ARCHIVOS is None:
    rutas = sorted(p for p in historico_path.rglob("*") if p.suffix.lower() in EXTENSIONES)
else:
    rutas = [historico_path / nombre for nombre in ARCHIVOS]

if not rutas:
    raise FileNotFoundError(
        "No se han encontrado archivos en la carpeta del histórico. "
        "Comprueba la ruta y que los archivos son CSV o Excel."
    )

tablas = []
for numero, ruta in enumerate(rutas, start=1):
    tabla = leer_archivo(ruta)
    tabla["archivo_origen"] = f"archivo_{numero}"
    tablas.append(tabla)

datos = pd.concat(tablas, ignore_index=True)
print("Archivos leídos:", len(rutas))
print("Filas totales:", len(datos))

datos = datos[datos[COL_SONDA].notna()].copy()

personales = [c for c in datos.columns
              if any(p in str(c).lower() for p in ["telegram", "agricultor", "contenido", "rol", "createdtime"])]
datos = datos.drop(columns=personales).dropna(axis=1, how="all")

print("Lecturas de sonda:", len(datos))
print("Columnas que se conservan:", list(datos.columns))

## 3. Estructura de los datos

Resumen de filas, columnas, tipos y valores vacíos. Sirve para documentar el histórico (tarea 1 de la issue) y para comprobar que las columnas importantes se han leído bien.

**Cómo interpretarlo:** una columna de humedad con tipo texto (`object`) suele indicar decimales con coma o valores como «N/D» mezclados; el apartado 4 lo corrige.

In [ ]:
resumen_columnas = pd.DataFrame({
    "tipo": datos.dtypes.astype(str),
    "valores_vacios": datos.isna().sum(),
    "porcentaje_vacios": (datos.isna().mean() * 100).round(1),
})
print("Filas y columnas:", datos.shape)
display(resumen_columnas)

## 4. Preparación: columnas, fechas, números y alias de las sondas

Este apartado:

1. Identifica las columnas de fecha, humedad, temperatura del suelo y sonda por su nombre (o usa las indicadas en la configuración).
2. Convierte las fechas, aceptando el formato día/mes/año con hora.
3. Convierte la humedad y la temperatura a números, aceptando la coma decimal.
4. Sustituye los identificadores de las sondas por alias, fusiona los que son una extensión incorrecta de otro (`FUSIONES_ALIAS`) y guarda la correspondencia **solo en la carpeta privada**.

**Qué comprobar:** que las columnas detectadas son las correctas y que pocas filas se pierden por una fecha o un número no interpretables.

In [ ]:
def detectar_columna(tabla, palabras, excluir=()):
    """Devuelve la primera columna cuyo nombre contenga alguna de las palabras."""
    for columna in tabla.columns:
        nombre = str(columna).lower()
        if any(p in nombre for p in palabras) and not any(e in nombre for e in excluir):
            return columna
    return None
col_fecha = COL_FECHA or detectar_columna(datos, ["fecha", "date", "timestamp"], excluir=["created", "creado"])
col_hum = COL_HUMEDAD or detectar_columna(datos, ["humedad", "vwc", "moisture"], excluir=["aire"])
col_temp = COL_TEMPERATURA or detectar_columna(datos, ["temp"], excluir=["aire"])
col_sonda = COL_SONDA or detectar_columna(datos, ["sonda id", "sonda", "probe"])

print("Columna de fecha:", col_fecha)
print("Columna de humedad:", col_hum)
print("Columna de temperatura del suelo:", col_temp)
print("Columna de sonda:", col_sonda)

if col_fecha is None or col_hum is None:
    raise ValueError("No se han podido identificar la columna de fecha o la de humedad. Indícalas en la configuración.")

def a_numero(serie):
    """Convierte a número aceptando la coma decimal."""
    if serie.dtype == object:
        serie = serie.astype(str).str.replace(",", ".", regex=False)
    return pd.to_numeric(serie, errors="coerce")

lecturas = pd.DataFrame({
    "fecha": pd.to_datetime(datos[col_fecha], dayfirst=True, errors="coerce"),
    "humedad": a_numero(datos[col_hum]),
    "archivo_origen": datos["archivo_origen"],
})
if col_temp is not None:
    lecturas["temperatura_suelo"] = a_numero(datos[col_temp])

if col_sonda is not None:
    identificadores = sorted(datos[col_sonda].dropna().astype(str).unique())
else:
    identificadores = ["sonda_unica"]
    datos["sonda_unica"] = "sonda_unica"
    col_sonda = "sonda_unica"

alias_inicial = {ident: f"sonda_{i}" for i, ident in enumerate(identificadores, start=1)}

# Se fusionan los identificadores que son una extensión incorrecta de otro.
alias_inicial = {ident: FUSIONES_ALIAS.get(a, a) for ident, a in alias_inicial.items()}

# Se renumeran para que los alias finales sean consecutivos.
orden = sorted(set(alias_inicial.values()), key=lambda a: int(a.split("_")[1]))
renumerar = {a: f"sonda_{i}" for i, a in enumerate(orden, start=1)}
alias = {ident: renumerar[a] for ident, a in alias_inicial.items()}
print("Equivalencia con los alias antes de fusionar:", renumerar)
lecturas["sonda"] = datos[col_sonda].astype(str).map(alias)

# La correspondencia entre identificador y alias queda solo en la carpeta privada.
privado_path.mkdir(parents=True, exist_ok=True)
(privado_path / "correspondencia_alias_sondas.json").write_text(
    json.dumps(alias, ensure_ascii=False, indent=2), encoding="utf-8"
)

invalidas = lecturas["fecha"].isna().sum()
sin_humedad = lecturas["humedad"].isna().sum()
print("Sondas distintas:", len(set(alias.values())))
print("Filas con fecha no interpretable:", int(invalidas))
print("Filas con humedad vacía o no numérica:", int(sin_humedad))

lecturas = lecturas.dropna(subset=["fecha"]).sort_values(["sonda", "fecha"]).reset_index(drop=True)

# Lecturas repetidas (misma sonda y misma fecha): se comprueba que son copias exactas y se conserva una.
repetidas = lecturas[lecturas.duplicated(subset=["sonda", "fecha"], keep=False)]
pares_rep = repetidas.groupby(["sonda", "fecha"])["humedad"].nunique()
print("Pares de lecturas repetidas:", len(pares_rep), "| con valores distintos:", int((pares_rep > 1).sum()))
antes = len(lecturas)
lecturas = lecturas.drop_duplicates(subset=["sonda", "fecha", "humedad"]).reset_index(drop=True)
print("Filas eliminadas por ser copias exactas:", antes - len(lecturas), "| lecturas que quedan:", len(lecturas))


## 5. Sondas y cobertura temporal

Para cada sonda: cuántas lecturas tiene, de qué fecha a qué fecha va y cuántos días distintos cubre. Responde a «¿qué sondas hay y qué periodo abarcan?».

**Qué comprobar:** que hay una sonda por parcela y que todas llegan, como mínimo, hasta el final del periodo del estudio. Si el histórico no trae ninguna columna de profundidad, la profundidad de 25 cm es la decisión de alcance del proyecto y se confirmará con los tutores.

In [ ]:
por_sonda = lecturas.groupby("sonda").agg(
    lecturas=("humedad", "size"),
    primera_lectura=("fecha", "min"),
    ultima_lectura=("fecha", "max"),
    dias_con_lecturas=("fecha", lambda s: s.dt.normalize().nunique()),
)
display(por_sonda)

## 6. Frecuencia de lectura y huecos

Se calcula el tiempo entre lecturas consecutivas de cada sonda. Se resume con la mediana y con los valores más pequeños y más grandes, y se listan los **huecos**: intervalos mayores que `HUECO_HORAS`.

**Cómo interpretarlo:** si la mediana es de unas pocas horas, el histórico permite ver el ciclo diario de la humedad. Los huecos largos son tramos sin información; no se rellenan ni se interpolan.

In [ ]:
lecturas["intervalo_h"] = (
    lecturas.groupby("sonda")["fecha"].diff().dt.total_seconds() / 3600
)

frecuencia = lecturas.groupby("sonda")["intervalo_h"].describe(percentiles=[0.05, 0.5, 0.95]).round(2)
display(frecuencia[["count", "min", "5%", "50%", "95%", "max"]])

huecos = lecturas[lecturas["intervalo_h"] > HUECO_HORAS].copy()
huecos["inicio_hueco"] = (huecos["fecha"] - pd.to_timedelta(huecos["intervalo_h"], unit="h")).dt.round("s")
huecos = huecos.rename(columns={"fecha": "fin_hueco"})[["sonda", "inicio_hueco", "fin_hueco", "intervalo_h"]]
print("Huecos de más de", HUECO_HORAS, "horas:", len(huecos))
display(huecos.sort_values("intervalo_h", ascending=False).head(15))

## 7. Valores vacíos y atípicos

Se marcan, sin borrar nada, las lecturas sospechosas:

- **Humedad vacía.**
- **Imposible físicamente:** menor o igual que 0, o mayor que 100.
- **Salto brusco:** cambio mayor que `SALTO_MAX` puntos entre dos lecturas consecutivas (criterio exploratorio).
- **Duplicada:** misma sonda y misma fecha repetida.

Queda **pendiente** la regla del punto de marchitez permanente: el proyecto decidió tratar una lectura por debajo del PMP como posible error, pero todavía no hay datos de textura para calcularlo. Cuando existan, esa marca se añadirá aquí.

In [ ]:
lecturas["dif_humedad"] = lecturas.groupby("sonda")["humedad"].diff()

marcas = pd.DataFrame({
    "vacia": lecturas["humedad"].isna(),
    "imposible": (lecturas["humedad"] <= 0) | (lecturas["humedad"] > 100),
    "salto_brusco": lecturas["dif_humedad"].abs() > SALTO_MAX,
    "duplicada": lecturas.duplicated(subset=["sonda", "fecha"], keep=False),
})
lecturas = pd.concat([lecturas, marcas], axis=1)

resumen_calidad = lecturas.groupby("sonda")[list(marcas.columns)].sum().astype(int)
resumen_calidad["lecturas"] = lecturas.groupby("sonda").size()
display(resumen_calidad)

estadisticos = lecturas.loc[~lecturas["imposible"] & ~lecturas["vacia"]].groupby("sonda")["humedad"].describe().round(2)
print("Estadísticos de la humedad (sin las lecturas imposibles):")
display(estadisticos)

### Comprobación de lecturas sospechosas

Muestra las lecturas imposibles y los saltos bruscos, y cuenta las lecturas sin temperatura por sonda. No se borra ni se corrige nada. Las lecturas repetidas ya se trataron en el apartado 4.


In [ ]:
sospechosas_tabla = lecturas.loc[lecturas["imposible"] | lecturas["salto_brusco"],
                                 ["sonda", "fecha", "humedad", "dif_humedad"]]
display(sospechosas_tabla)
print("Lecturas sin temperatura por sonda:")
display(lecturas.groupby("sonda")["temperatura_suelo"].apply(lambda s: int(s.isna().sum())))


## 8. Solape con el periodo del estudio

Compara los días con lecturas de cada sonda con los días del periodo provisional del estudio (de `PERIODO_INICIO`, incluido, a `PERIODO_FIN`, excluido). La fecha de fin es futura, por lo que los días posteriores a la última lectura se cuentan como sin datos.

**Cómo interpretarlo:** el porcentaje de cobertura indica cuánto del periodo se puede analizar. Un valor bajo en una sonda significa que esa parcela tendrá un tramo sin humedad medida.

In [ ]:
dias_periodo = pd.date_range(PERIODO_INICIO, PERIODO_FIN, inclusive="left")
lecturas_validas = lecturas[~lecturas["vacia"] & ~lecturas["imposible"]]

filas = []
for sonda, tabla in lecturas_validas.groupby("sonda"):
    dias = pd.DatetimeIndex(tabla["fecha"].dt.normalize().unique())
    dias_cubiertos = dias.intersection(dias_periodo)
    filas.append({
        "sonda": sonda,
        "dias_del_periodo": len(dias_periodo),
        "dias_con_datos": len(dias_cubiertos),
        "cobertura_pct": round(100 * len(dias_cubiertos) / len(dias_periodo), 1),
    })
solape = pd.DataFrame(filas).set_index("sonda")
display(solape)

## 9. Episodios de subida de humedad (exploratorio)

Una subida brusca de la humedad suele corresponder a un riego o a una lluvia. Aquí se buscan **candidatos**: momentos en los que la humedad sube al menos `UMBRAL_SUBIDA` puntos respecto al mínimo de las `VENTANA_SUBIDA_H` horas anteriores. Las lecturas contiguas se agrupan en un episodio.

Para cada episodio se indica el inicio, el pico, la amplitud y cuántas lecturas hay en la ventana posterior al pico (de 24 a 48 h). Esa ventana es la que usaría la autocalibración de la capacidad de campo.

**Importante:** esto no calibra nada. Los umbrales reales de la calibración (subida y drenaje) están pendientes de los tutores. Los episodios no se confirman como riego o lluvia hasta cruzarlos con los registros de riego y con la meteorología.

In [ ]:
def detectar_episodios(tabla):
    """Detecta episodios de subida en una sonda y devuelve una tabla."""
    serie = tabla.set_index("fecha")["humedad"].dropna()
    serie = serie[~serie.index.duplicated()]
    if serie.empty:
        return []
    minimo_previo = serie.rolling(f"{VENTANA_SUBIDA_H}h").min()
    subida = serie - minimo_previo
    marcados = subida[subida >= UMBRAL_SUBIDA]
    if marcados.empty:
        return []

    episodios = []
    inicio = ultimo = marcados.index[0]
    for momento in marcados.index[1:]:
        if (momento - ultimo) > pd.Timedelta(hours=SEPARACION_EPISODIOS_H):
            episodios.append((inicio, ultimo))
            inicio = momento
        ultimo = momento
    episodios.append((inicio, ultimo))

    filas = []
    for inicio, fin in episodios:
        tramo = serie[inicio - pd.Timedelta(hours=VENTANA_SUBIDA_H):fin]
        pico_momento = tramo.idxmax()
        base = tramo.min()
        post = serie[pico_momento + pd.Timedelta(hours=VENTANA_POST_PICO_H[0]):
                     pico_momento + pd.Timedelta(hours=VENTANA_POST_PICO_H[1])]
        filas.append({
            "inicio": inicio,
            "pico": pico_momento,
            "amplitud_puntos": round(tramo.max() - base, 2),
            "horas_hasta_pico": round((pico_momento - tramo.idxmin()).total_seconds() / 3600, 1),
            "lecturas_en_ventana_post": len(post),
        })
    return filas

episodios_todos = []
for sonda, tabla in lecturas_validas.groupby("sonda"):
    for fila in detectar_episodios(tabla):
        fila["sonda"] = sonda
        episodios_todos.append(fila)

episodios = pd.DataFrame(episodios_todos)
if episodios.empty:
    print("No se han detectado episodios con los parámetros actuales.")
else:
    episodios = episodios[["sonda", "inicio", "pico", "amplitud_puntos", "horas_hasta_pico", "lecturas_en_ventana_post"]]
    print("Episodios candidatos por sonda:")
    display(episodios.groupby("sonda").size().rename("episodios"))
    display(episodios.sort_values(["sonda", "inicio"]).head(20))

## 10. Figuras

Una figura por sonda con la humedad a lo largo del tiempo. La línea se **corta en los huecos** para que se vean los tramos sin datos, y las lecturas imposibles o con salto brusco se marcan en rojo. Las figuras se guardan en la carpeta privada de resultados, no en el repositorio.

**Cómo leerlas:** busca el ciclo diario, las subidas bruscas (candidatas del apartado 9) y los tramos planos o sin datos.

In [ ]:
for sonda, tabla in lecturas.groupby("sonda"):
    tabla = tabla[~tabla["vacia"]].sort_values("fecha")
    if tabla.empty:
        print(f"{sonda}: sin lecturas válidas")
        continue

    # Cada tramo continuo (sin huecos largos) se dibuja como una línea.
    tramo_id = (tabla["intervalo_h"] > HUECO_HORAS).cumsum()

    fig, ax = plt.subplots(figsize=(11, 3.5))
    for _, tramo in tabla.groupby(tramo_id):
        ax.plot(tramo["fecha"], tramo["humedad"], linewidth=0.8, color="tab:blue")
    ax.scatter(tabla["fecha"], tabla["humedad"], s=6, color="tab:blue", label="Humedad")

    sospechosas = tabla[tabla["imposible"] | tabla["salto_brusco"]]
    ax.scatter(sospechosas["fecha"], sospechosas["humedad"], color="red", s=14,
               label="Lectura sospechosa", zorder=3)
    ax.axvspan(pd.Timestamp(PERIODO_INICIO), pd.Timestamp(PERIODO_FIN),
               color="grey", alpha=0.08, label="Periodo del estudio")
    ax.set_title(f"Humedad medida por la {sonda.replace('_', ' ')}")
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Humedad (% vol.)")
    ax.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    fig.savefig(resultados_path / f"humedad_{sonda}.png", dpi=130)
    plt.show()

## 11. Exportación de las tablas

Se guardan en la carpeta privada de resultados las tablas del análisis, con los alias de las sondas. Son los datos que se resumirán en `docs/analisis-historico-sondas.md`.

In [ ]:
resumen_columnas.to_csv(resultados_path / "estructura_columnas.csv", encoding="utf-8-sig")
por_sonda.to_csv(resultados_path / "cobertura_por_sonda.csv", encoding="utf-8-sig")
frecuencia.to_csv(resultados_path / "frecuencia_lecturas.csv", encoding="utf-8-sig")
huecos.to_csv(resultados_path / "huecos.csv", index=False, encoding="utf-8-sig")
resumen_calidad.to_csv(resultados_path / "calidad_lecturas.csv", encoding="utf-8-sig")
solape.to_csv(resultados_path / "solape_periodo.csv", encoding="utf-8-sig")
if not episodios.empty:
    episodios.to_csv(resultados_path / "episodios_subida.csv", index=False, encoding="utf-8-sig")

print("Tablas y figuras guardadas en:", resultados_path)

## 12. Cierre y siguientes pasos

Con las tablas de este cuaderno se redacta `docs/analisis-historico-sondas.md`: estructura, sondas, frecuencia, huecos, calidad, cobertura del periodo y episodios candidatos.

Pendiente para las tareas siguientes:

- Confirmar con los tutores las unidades y la profundidad de las sondas.
- Aplicar la regla del PMP cuando haya datos de textura.
- Cruzar los episodios de subida con el registro de riegos y la meteorología (issues #11 y #13).
- Definir los umbrales de la autocalibración de la capacidad de campo.

**Antes de publicar este cuaderno:** borrar todas las salidas y comprobar que no contiene identificadores de sondas, nombres de archivos reales ni rutas privadas.